# Step 1 — Compute clean per-neuron spike metrics

| | |
|---|---|
| **Original** | `Liam_work/tests/non_behavioral_spike_analysis/compute_clean_spikemetrics.ipynb` |
| **Status** | **Pipeline step 1.** Run once; feeds the XGBoost notebook (step 2). |

**What it does.** Loads the raw per-neuron ISI table (`all_animal_isi_df.pkl`, 13 GB, 5,501 neurons × 20
animals; one list of per-epoch ISI arrays per neuron per state), drops non-positive / non-finite ISIs
(a small number of negative ISIs exist in the raw data), and recomputes
12 metrics per neuron **separately for Wake, NREM and REM**:

| metric | meaning |
|---|---|
| `mean_isis`, `median_isis`, `max_isis`, `min_isis` | ISI location / extremes (s) |
| `mean_ifr` | 1 / mean ISI (Hz) |
| `sd`, `cv` | spread; CV = SD / mean |
| `cv2`, `lv` | local (adjacent-ISI) irregularity (elephant) |
| `lvr` | revised local variation, refractory R = 5 ms (elephant) |
| `dist_alpha`, `dist_beta` | shape / scale of a gamma fit to the ISIs |

**Why it exists.** Ben's earlier `*_spkmetrics.pkl` tables were built from unfiltered ISIs (negatives
corrupted `min_isis` etc.) and LvR was effectively a duplicate of LV. These tables replace them.

**Output.** `data/spike_metrics/clean_spikemetrics_{nrem,wake,rem}.pkl` — one row per neuron, rows aligned
across the three files, tagged `attrs['metric_schema'] = 'isi-v2-lvr5ms'` (the XGBoost notebook refuses
files without this tag). The copies committed in `data/spike_metrics/` were produced by Liam's run.

**Runtime.** ~25 min (gamma fits dominate; 32 joblib workers × 1 BLAS thread).

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

In [ ]:
# Shared-server defaults; set before importing numerical libraries.
import os
for _thread_var in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                    'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[_thread_var] = '1'
import numpy as np
import pandas as pd
import pickle
import os
import elephant.statistics
from scipy import stats
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import warnings
import time

tqdm.pandas()
# Keep numerical and compatibility warnings visible.

ISI_DIR = '/media/HlabShare/Ben_work/isi/'
# Repo copy: outputs go to the project's data/ folder (Liam's original wrote next to the notebook).
PROJECT_DIR = '/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project'
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'data', 'spike_metrics')
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_JOBS = 32  # 32 workers x 1 BLAS thread each (thread vars above are '1', not the usual '4',
             # so total CPU use stays at 32 cores on marvin)

print(f'Output directory: {OUTPUT_DIR}')
print(f'Parallel workers: {N_JOBS}')
print('Imports loaded.')

## 1. Load Raw ISI Data

In [ ]:
raw_path = os.path.join(ISI_DIR, 'all_animal_isi_df.pkl')
print(f'Loading {raw_path} ...', flush=True)
t0 = time.time()

with open(raw_path, 'rb') as f:
    raw_df = pickle.load(f)

print(f'Loaded in {time.time() - t0:.1f}s')
print(f'Shape: {raw_df.shape}')
print(f'Columns: {list(raw_df.columns)}')
print(f'Animals: {raw_df["animal"].nunique()}')
print(f'\nFirst few rows (metadata only):')
meta_cols = [c for c in raw_df.columns if '_isi' not in c]
print(raw_df[meta_cols].head())

## 2. Define Metric Computation Functions

Replicates Ben's pipeline from `pre-processing_splitting.ipynb` with the addition of negative ISI filtering.

In [ ]:
# =============================================================================
# Metric functions based on the historical pipeline, with positive finite
# ISI filtering and an explicit, corrected LvR refractory period of 5 ms.
# =============================================================================

def clean_hstack(isi_list):
    """Concatenate epoch ISI arrays (like Ben's np.hstack) and filter negatives."""
    if not isinstance(isi_list, list):
        return np.array([])
    try:
        isis = np.hstack(isi_list)
    except:
        return np.array([])
    isis = isis.astype(float)
    mask = np.isfinite(isis) & (isis > 0)
    return isis[mask]


def compute_cv2(x, with_nan=False):
    if len(x) <= 1:
        return np.nan
    return elephant.statistics.cv2(x, with_nan)

def compute_lv(x, with_nan=False):
    if len(x) <= 1:
        return np.nan
    return elephant.statistics.lv(x, with_nan)

def compute_lvr(x, with_nan=False):
    if len(x) <= 1:
        return np.nan
    # it assumes milliseconds
    return elephant.statistics.lvr(x * 1000, R=5.0, with_nan=with_nan)

def compute_std(x):
    if len(x) == 0:
        return np.nan
    return np.std(x)

def compute_mean(x):
    if len(x) == 0:
        return np.nan
    return np.mean(x)

def compute_ifr(x):
    if len(x) == 0:
        return np.nan
    return 1 / np.mean(x)

def compute_cv(x, with_nan=False):
    if len(x) == 0:
        return np.nan
    return np.std(x) / np.mean(x)

def compute_median(x):
    if len(x) == 0:
        return np.nan
    return np.median(x)

def compute_min(x):
    if len(x) == 0:
        return np.nan
    return np.min(x)

def compute_max(x):
    if len(x) == 0:
        return np.nan
    return np.max(x)

def fit_gamma(x):
    """Fit gamma distribution — matches Ben's fit_gamma exactly."""
    try:
        alpha, loc, beta = stats.gamma.fit(x)
    except:
        alpha, loc, beta = None, None, None
    return alpha, beta


print('Metric functions defined: positive finite ISIs; explicit LvR R=5 ms.')

## 3. Compute Clean Metrics Per State

For each sleep state (NREM, Wake, REM):
1. Extract raw ISI arrays per neuron
2. Filter out negative/invalid ISIs
3. Compute all 12 metrics
4. Save as a clean DataFrame

In [ ]:
META_COLS = ['animal', 'epoch', 'cell_no', 'age', 'sex', 'geno', 'brain_region', 'cell_type']
METRIC_NAMES = ['mean_isis', 'mean_ifr', 'median_isis', 'max_isis', 'min_isis',
                'sd', 'cv', 'cv2', 'lv', 'lvr', 'dist_alpha', 'dist_beta']

state_isi_cols = {'nrem': 'nrem_isi', 'wake': 'wake_isi', 'rem': 'rem_isi'}

for state, isi_col in state_isi_cols.items():
    print(f'\n{"="*60}')
    print(f'Processing {state.upper()}')
    print(f'{"="*60}')
    t0 = time.time()
    out_path = os.path.join(OUTPUT_DIR, f'clean_spikemetrics_{state}.pkl')

    # Work on a copy with just this state's ISIs
    state_df = raw_df[META_COLS + [isi_col]].copy()

    # Step 1: Concatenate ISI arrays and filter negatives (like Ben's np.hstack + our filter)
    print('Concatenating & filtering ISIs...')
    raw_counts = state_df[isi_col].apply(
        lambda x: sum(len(a) for a in x if isinstance(a, np.ndarray)) if isinstance(x, list) else 0)
    state_df[isi_col] = state_df[isi_col].progress_apply(clean_hstack)
    clean_counts = state_df[isi_col].apply(len)
    total_raw = raw_counts.sum()
    total_clean = clean_counts.sum()
    total_filtered = total_raw - total_clean
    print(f'  Filtered {total_filtered:,} invalid ISIs out of {total_raw:,} '
          f'({total_filtered/max(total_raw,1)*100:.2f}%)')

    # Step 2: Compute each metric with progress_apply (matching Ben's pattern)
    print('Mean ISI')
    state_df['mean_isis'] = state_df[isi_col].progress_apply(compute_mean)

    print('Mean IFR')
    state_df['mean_ifr'] = state_df['mean_isis'].progress_apply(lambda x: 1/x if x > 0 else np.nan)

    print('Median ISI')
    state_df['median_isis'] = state_df[isi_col].progress_apply(compute_median)

    print('Max ISI')
    state_df['max_isis'] = state_df[isi_col].progress_apply(compute_max)

    print('Min ISI')
    state_df['min_isis'] = state_df[isi_col].progress_apply(compute_min)

    print('Std Dev')
    state_df['sd'] = state_df[isi_col].progress_apply(compute_std)

    print('CV')
    state_df['cv'] = state_df[isi_col].progress_apply(compute_cv)

    print('CV2')
    state_df['cv2'] = state_df[isi_col].progress_apply(compute_cv2)

    print('LV')
    state_df['lv'] = state_df[isi_col].progress_apply(compute_lv)

    print('LVr')
    state_df['lvr'] = state_df[isi_col].progress_apply(compute_lvr)

    # Step 3: Gamma fit — parallelized with joblib (this is the slow one)
    print(f'Gamma fit (parallel, {N_JOBS} workers)...')
    gamma_results = Parallel(n_jobs=N_JOBS, backend='loky')(
        delayed(fit_gamma)(isis) for isis in tqdm(state_df[isi_col], desc='gamma fit')
    )
    state_df['dist_alpha'] = [r[0] for r in gamma_results]
    state_df['dist_beta'] = [r[1] for r in gamma_results]

    # Save
    out_df = state_df[META_COLS + METRIC_NAMES].copy()
    out_df.attrs.update(metric_schema='isi-v2-lvr5ms', lvr_R_ms=5.0)
    out_df.to_pickle(out_path)

    elapsed = time.time() - t0
    print(f'\n  Done in {elapsed:.1f}s')
    print(f'  Neurons with <2 valid ISIs: {(clean_counts < 2).sum()}')
    print(f'  Saved: {out_path}')
    print(f'  File size: {os.path.getsize(out_path)/1024/1024:.1f} MB')

    del state_df  # free memory

print(f'\n{"="*60}')
print('ALL STATES COMPLETE')
print(f'{"="*60}')

## 4. Validate Output

In [ ]:
# Reload and validate saved files
for state in ['nrem', 'wake', 'rem']:
    path = os.path.join(OUTPUT_DIR, f'clean_spikemetrics_{state}.pkl')
    df = pd.read_pickle(path)
    
    assert df.attrs.get('metric_schema') == 'isi-v2-lvr5ms'
    print(f'\n{state.upper()}: {df.shape}')
    print(f'  Animals: {df["animal"].nunique()}')
    
    # Check min_isis is always positive (the key fix)
    valid = df['min_isis'].dropna()
    neg = (valid < 0).sum()
    print(f'  min_isis: min={valid.min():.6f}, max={valid.max():.6f}, negatives={neg}')
    
    # Check NaN counts per metric
    nan_counts = df[METRIC_NAMES].isna().sum()
    print(f'  NaN counts per metric:')
    for col, n in nan_counts.items():
        if n > 0:
            print(f'    {col}: {n}')
    if nan_counts.sum() == 0:
        print(f'    (none)')
    
    # Compare with original spkmetrics
    orig_path = os.path.join(ISI_DIR, f'all_animal_isi_df_{state}_spkmetrics.pkl')
    if os.path.exists(orig_path):
        orig_df = pd.read_pickle(orig_path)
        shared_cols = [c for c in METRIC_NAMES if c in orig_df.columns]
        print(f'  Comparison with original spkmetrics ({len(shared_cols)} shared metrics):')
        for col in ['mean_isis', 'min_isis', 'cv', 'lv']:
            if col in shared_cols:
                corr = df[col].corr(orig_df[col])
                print(f'    {col}: correlation = {corr:.4f}')

print(f'\nValidation complete. Files ready for use in xgboost_spike_classification.ipynb')